<a href="https://colab.research.google.com/github/aycakrk/DI725_Ayca/blob/main/Final_Project/phase3_qlora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!unzip "/content/drive/MyDrive/RISCM.zip" -d "/content"

Streaming output truncated to the last 5000 lines.
  inflating: /content/resized/NWPU_1168.jpg  
  inflating: /content/resized/RSICD_4364.jpg  
  inflating: /content/resized/NWPU_18440.jpg  
  inflating: /content/resized/NWPU_129.jpg  
  inflating: /content/resized/NWPU_19322.jpg  
  inflating: /content/resized/RSICD_3101.jpg  
  inflating: /content/resized/UCM_2077.jpg  
  inflating: /content/resized/NWPU_10992.jpg  
  inflating: /content/resized/NWPU_10941.jpg  
  inflating: /content/resized/NWPU_19048.jpg  
  inflating: /content/resized/UCM_1535.jpg  
  inflating: /content/resized/NWPU_22896.jpg  
  inflating: /content/resized/NWPU_15332.jpg  
  inflating: /content/resized/RSICD_7475.jpg  
  inflating: /content/resized/NWPU_24431.jpg  
  inflating: /content/resized/NWPU_25180.jpg  
  inflating: /content/resized/RSICD_1536.jpg  
  inflating: /content/resized/NWPU_18103.jpg  
  inflating: /content/resized/NWPU_906.jpg  
  inflating: /content/resized/NWPU_11526.jpg  
  inflating: /cont

In [1]:
# Örnek: captions.csv'yi oku
import pandas as pd
df = pd.read_csv('captions.csv')
df.head()


,source,split,image,caption_1,caption_2,caption_3,caption_4,caption_5
0,NWPU,test,NWPU_31430.jpg,A gray plane on the runway and the lawn beside .,A grey plane is on the runway by the lawn .,There is an airplane on the runway with a larg...,A plane is parked on the runway next to the gr...,There is a plane on the runway beside the grass .
1,NWPU,test,NWPU_31431.jpg,Three small planes parked in a line on the air...,"There are four aircraft on the open ground, Th...",There are many planes of different sizes in a ...,Four planes are parked on the runway .,Four planes of different sizes were on the mar...
2,NWPU,test,NWPU_31432.jpg,A plane parked in a line on the airport with s...,A white plane was parked on the instruction li...,An airplane parked in an open area with many c...,A plane is parked on the open space .,There is 1 plane on the ground marked .
3,NWPU,test,NWPU_31433.jpg,A small plane and a big plane parked next to b...,A white plane and a gray plane parked at the b...,Two planes of different sizes are neatly parke...,A large plane and a small plane are parked nea...,Two planes are on the marked ground .
4,NWPU,test,NWPU_31434.jpg,Two planes parked next to boarding bridges .,Two aircraft were parked at the departure gates .,Two planes of different sizes are neatly parke...,Two planes are parked next to the terminal .,Two planes are on the marked ground .


In [4]:
!pip install huggingface_hub
!pip install evaluate transformers tqdm wandb
!pip install rouge_score
!pip install pycocoevalcap


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 8.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=34c92bd4e8b3d6aaf72c7c2569aebf8811c7c48478c6408b090dfad40dab0a90
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 24.3 MB/s eta 0:00:00


In [2]:
from huggingface_hub import notebook_login
notebook_login()

In [3]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

train_df = df[df["split"] == "train"].copy()
val_df   = df[df["split"] == "val"].copy()
test_df  = df[df["split"] == "test"].copy()

# length evaluation
n_train = len(train_df)
n_val   = len(val_df)
n_test  = len(test_df)
total   = n_train + n_val + n_test

print(f"Train: {n_train} ({n_train/total:.2%})")
print(f"Test:  {n_test}  ({n_test/total:.2%})")
print(f"Val:   {n_val}   ({n_val/total:.2%})")

img_dir = '/content/resized'
all_images = set(os.listdir(img_dir))
df_images  = set(df['image'].unique())

missing = sorted(list(df_images - all_images))
if missing:
    print(f"Eksik {len(missing)} dosya var. Örnekler:", missing[:10])
else:
    print("Tüm caption’daki image isimleri klasörde bulundu.")


Train: 35614 (79.99%)
Test:  4454  (10.00%)
Val:   4453   (10.00%)
Tüm caption’daki image isimleri klasörde bulundu.


In [8]:
import os
import random
from PIL import Image
import wandb
import pandas as pd
import torch
from torch.utils.data import Dataset

from transformers import (
    PaliGemmaForConditionalGeneration,
    AutoProcessor,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    default_data_collator,
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate
from pycocoevalcap.cider.cider import Cider

# -------------------------------------------
# 0) WANDB oturumu
# -------------------------------------------
wandb.init(project="DI725_FinalProject", name="lora_r8_eos_post")

# -------------------------------------------
# 1) Model ve Processor Yükleme
# -------------------------------------------
model_name = "google/paligemma-3b-pt-224"
processor = AutoProcessor.from_pretrained(model_name, use_auth_token=True)
tokenizer = processor.tokenizer  # EOS token vs. için

base_model = PaliGemmaForConditionalGeneration.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16,  # fp16 da olur ama bf16 daha stabil
    use_auth_token=True
)

# Dondurulmayacak kısımlar dışında her şey freeze
for param in base_model.vision_tower.parameters():
    param.requires_grad = False

for param in base_model.multi_modal_projector.parameters():
    param.requires_grad = False

# -------------------------------------------
# 2) Dinamik Caption Dataset
# -------------------------------------------
class RiscDynamicCaptionDataset(Dataset):
    def __init__(self, df, img_dir, processor, max_len=256):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.proc = processor
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row["image"])).convert("RGB")
        caps = [row[f"caption_{i}"] for i in range(1, 6)]
        cap = "<image> " + random.choice(caps) + tokenizer.eos_token

        enc = self.proc(
            images=img,
            text=cap,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "pixel_values": enc.pixel_values.squeeze(0),
            "input_ids": enc.input_ids.squeeze(0),
            "attention_mask": enc.attention_mask.squeeze(0),
            "labels": enc.input_ids.squeeze(0).clone()
        }

# -------------------------------------------
# 3) Train / Val Dataset’leri yükle
# -------------------------------------------
img_dir = "/content/resized"
train_ds = RiscDynamicCaptionDataset(train_df, img_dir, processor)
val_ds   = RiscDynamicCaptionDataset(val_df,   img_dir, processor)

# -------------------------------------------
# 4) LoRA Konfigürasyonu & PEFT Model
# -------------------------------------------
lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model_lora = get_peft_model(base_model, lora_cfg)
model_lora.print_trainable_parameters()

# -------------------------------------------
# 5) Eğitim Parametreleri & Trainer
# -------------------------------------------
training_args = Seq2SeqTrainingArguments(
    output_dir="lora_r8",
    per_device_train_batch_size=3,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    num_train_epochs=1,
    logging_steps=5,
    eval_strategy="epoch",
    logging_strategy="steps",
    predict_with_generate=True,
    bf16=True,  # varsa bf16, yoksa fp16 kullanabilirsin
    push_to_hub=False,
    report_to="wandb",
    max_steps=1500
)

trainer = Seq2SeqTrainer(
    model=model_lora,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=default_data_collator,
    tokenizer=processor
)

# -------------------------------------------
# 6) Eğitim & Değerlendirme
# -------------------------------------------
trainer.train()
lora_metrics = trainer.evaluate()
print("LoRA metrics:", lora_metrics)

# -------------------------------------------
# 7) Tahmin & Değerlendirme Metrikleri
# -------------------------------------------
raw_pred_ids = trainer.predict(val_ds).predictions
clean_preds = [tokenizer.decode(ids, skip_special_tokens=True).strip() for ids in raw_pred_ids]

references = val_df[[f"caption_{i}" for i in range(1,6)]].values.tolist()

# CIDEr
refs_dict  = {i: references[i] for i in range(len(references))}
preds_dict = {i: [clean_preds[i]] for i in range(len(clean_preds))}
cider_scorer = Cider()
cider_score, _ = cider_scorer.compute_score(refs_dict, preds_dict)

# BLEU, METEOR, ROUGE-L
bleu_res   = evaluate.load("bleu").compute(references=references, predictions=clean_preds)["bleu"]
meteor_res = evaluate.load("meteor").compute(references=references, predictions=clean_preds)["meteor"]
rouge_res  = evaluate.load("rouge").compute(references=references, predictions=clean_preds)["rougeL"]

print(f"BLEU:    {bleu_res:.4f}")
print(f"METEOR:  {meteor_res:.4f}")
print(f"ROUGE-L: {rouge_res:.4f}")
print(f"CIDEr:   {cider_score:.4f}")

wandb.log({
    "bleu":   bleu_res,
    "meteor": meteor_res,
    "rougeL": rouge_res,
    "cider":  cider_score
})

# -------------------------------------------
# 8) CSV ve WandB için örnek kayıt
# -------------------------------------------
rows = []
for i, img_name in enumerate(val_ds.df["image"].tolist()):
    row = {"image": img_name, "pred": clean_preds[i]}
    for j, r in enumerate(references[i], start=1):
        row[f"ref_{j}"] = r
    rows.append(row)
pd.DataFrame(rows).to_csv("lora_r8_eos_post_val_results.csv", index=False)
print("Saved CSV → lora_r8_eos_post_val_results.csv")

table = wandb.Table(columns=["image", "prediction", "references"])
for idx in random.sample(range(len(clean_preds)), 5):
    table.add_data(val_ds.df["image"].iloc[idx],
                   clean_preds[idx],
                   " || ".join(references[idx]))
wandb.log({"examples": table})


/usr/local/lib/python3.11/dist-packages/transformers/models/auto/processing_auto.py:255: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/699 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json:   0%|          | 0.00/40.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.26M [00:00<?, ?B/s]

...

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
0,13.059700,13.111368


LoRA metrics: {'eval_loss': 13.111368179321289, 'eval_runtime': 254.5174, 'eval_samples_per_second': 17.496, 'eval_steps_per_second': 2.188, 'epoch': 0.5053908355795148}


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


BLEU:    0.0000
METEOR:  0.0000
ROUGE-L: 0.0000
CIDEr:   0.0000
Saved CSV → lora_r8_eos_post_val_results.csv


In [4]:
!pip install -q -U bitsandbytes

In [5]:
!pip install -q -U transformers datasets peft bitsandbytes

In [6]:
import requests
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
import requests
from datasets import load_dataset
from peft import get_peft_model, LoraConfig
from transformers import BitsAndBytesConfig
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration

In [37]:
import os
import io
import random
import pandas as pd
import wandb
import torch
from PIL import Image
from datasets import Dataset
from transformers import (
    PaliGemmaProcessor,
    PaliGemmaForConditionalGeneration,
    Trainer,
    TrainingArguments,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model
import evaluate
from pycocoevalcap.cider.cider import Cider

# 0) WANDB oturumu
wandb.init(project="DI725_FinalProject", name="finetuned_paligemma")

# 1) Yardımcı: HF image‐feature → PIL dönüşümü
def to_pil(img):
    if isinstance(img, Image.Image):
        return img
    elif isinstance(img, dict) and img.get("path"):
        return Image.open(img["path"]).convert("RGB")
    elif isinstance(img, dict) and img.get("bytes"):
        return Image.open(io.BytesIO(img["bytes"])).convert("RGB")
    else:
        # numpy array vb.
        return Image.fromarray(img).convert("RGB")

# 2) Veriyi oku ve HF Dataset nesnesine dönüştür
df = pd.read_csv("captions.csv")
train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "val"].reset_index(drop=True)

def df_to_dataset(df):
    def load_image(example):
        example["image_id"] = example["image"]
        example["image"] = {"path": os.path.join("/content/resized", example["image"])}
        return example

    ds = Dataset.from_pandas(df)
    return ds.map(load_image)

train_ds = df_to_dataset(train_df)
val_ds   = df_to_dataset(val_df)

# 3) Processor ve model
model_id  = "google/paligemma-3b-pt-224"
processor = PaliGemmaProcessor.from_pretrained(model_id)

def collate_fn(examples):
    images, texts = [], []
    for ex in examples:
        pil = to_pil(ex["image"])
        images.append(pil)
        texts.append(f"<image> <bos> describe this image. {ex['caption_1']}")
    batch = processor(
        images=images,
        text=texts,
        return_tensors="pt",
        padding="longest",
        truncation=True
    )
    # Trainer’ın internal loss hesabı için labels ekle
    batch["labels"] = batch["input_ids"].clone()
    return batch

# 4) Model & LoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)
base_model = PaliGemmaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
lora_config = LoraConfig(
    r=8,
    target_modules=["q_proj","k_proj","v_proj","o_proj","up_proj","down_proj","gate_proj"],
    task_type="CAUSAL_LM"
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# 5) Eğitim argümanları
args = TrainingArguments(
    output_dir="finetuned_paligemma_riscm_yourdata",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    num_train_epochs=1,
    dataloader_pin_memory=False,
    bf16=True,
    logging_steps=10,
    max_steps=500,
    save_strategy="epoch",
    report_to="wandb",
    remove_unused_columns=False,
    dataloader_num_workers=8
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate_fn
)

# 6) Eğitim
trainer.train()

from time import time
from tqdm.auto import tqdm
import math
import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32   = True

# 1) Prepare model for fast generation
model.eval()
model.gradient_checkpointing_disable()   # re-enable use_cache

# 2) Inference in mini-batches with ETA
clean_preds, references = [], []
batch_size = 32
total_batches = math.ceil(len(val_ds) / batch_size)

pbar = tqdm(total=total_batches, desc="Generating", unit="batch")

for batch_idx in range(0, len(val_ds), batch_size):
    start_time = time()

    # select a true Dataset slice
    batch = val_ds.select(range(batch_idx, min(batch_idx+batch_size, len(val_ds))))
    images = [to_pil(ex["image"]) for ex in batch]
    refs   = [[ex[f"caption_{j}"] for j in range(1,6)] for ex in batch]
    prompts = ["<image> <bos> describe this image."] * len(images)

    inputs = processor(
        text=prompts,
        images=images,
        return_tensors="pt",
        padding="longest",
        truncation=True
    ).to("cuda")

    gen_ids = model.generate(**inputs, max_new_tokens=64)
    texts   = processor.tokenizer.batch_decode(gen_ids, skip_special_tokens=True)

    # record
    clean_preds.extend([t.strip() for t in texts])
    references.extend(refs)

    # update progress bar
    batch_time = time() - start_time
    pbar.update(1)
    pbar.set_postfix({"last_batch_s": f"{batch_time:.1f}"})

pbar.close()

# 3) Compute metrics (same as before)
refs_dict  = {i: references[i]    for i in range(len(references))}
preds_dict = {i: [clean_preds[i]] for i in range(len(clean_preds))}
cider_scorer = Cider()
cider_score, _ = cider_scorer.compute_score(refs_dict, preds_dict)

bleu   = evaluate.load("bleu"  ).compute(predictions=clean_preds, references=references)["bleu"]
meteor = evaluate.load("meteor").compute(predictions=clean_preds, references=references)["meteor"]
rouge  = evaluate.load("rouge" ).compute(predictions=clean_preds, references=references)["rougeL"]

metrics = {
    "eval/bleu":   bleu,
    "eval/meteor": meteor,
    "eval/rougeL": rouge,
    "eval/cider":  cider_score,
}
print(metrics)
wandb.log(metrics)

# 4) Save to CSV (same as before)
rows = []
for i, ex in enumerate(val_df.itertuples(index=False)):
    row = {"image": ex.image, "pred": clean_preds[i]}
    for j in range(1,6):
        row[f"ref_{j}"] = getattr(ex, f"caption_{j}")
    rows.append(row)
pd.DataFrame(rows).to_csv("inference_results.csv", index=False)


train/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
train/global_step,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇████
train/grad_norm,▅▇█▇▆▄▄▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/learning_rate,████▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁
train/loss,█▇▇▆▅▄▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
total_flos,1.624744280085024e+16
train/epoch,0.11232
train/global_step,500
train/grad_norm,0.40001
train/learning_rate,0.0
train/loss,12.1456


Map:   0%|          | 0/35614 [00:00<?, ? examples/s]

Map:   0%|          | 0/4453 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


...

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to tru

Step,Training Loss
10,19.686300
20,19.073800
30,18.369900
40,17.325800
50,16.650300
60,15.875700
70,15.328000
80,14.799900
90,14.302600
100,13.863600


Generating:   0%|          | 0/140 [00:00<?, ?batch/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


{'eval/bleu': 0.0, 'eval/meteor': np.float64(0.06911412588396391), 'eval/rougeL': np.float64(0.03238544158177521), 'eval/cider': np.float64(0.0009708322805579115)}


In [45]:
import os
import io
import random
import pandas as pd
import wandb
import torch
from PIL import Image
from datasets import Dataset
from transformers import (
    PaliGemmaProcessor,
    PaliGemmaForConditionalGeneration,
    Trainer,
    TrainingArguments,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model
import evaluate
from pycocoevalcap.cider.cider import Cider

# 0) WANDB oturumu
wandb.init(project="DI725_FinalProject", name="finetuned_paligemma")

# 1) Yardımcı: HF image‐feature → PIL dönüşümü
def to_pil(img):
    if isinstance(img, Image.Image):
        return img
    elif isinstance(img, dict) and img.get("path"):
        return Image.open(img["path"]).convert("RGB")
    elif isinstance(img, dict) and img.get("bytes"):
        return Image.open(io.BytesIO(img["bytes"])).convert("RGB")
    else:
        # numpy array vb.
        return Image.fromarray(img).convert("RGB")

# 2) Veriyi oku ve HF Dataset nesnesine dönüştür
df = pd.read_csv("captions.csv")
train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "val"].reset_index(drop=True)

def df_to_dataset(df):
    def load_image(example):
        example["image_id"] = example["image"]
        example["image"] = {"path": os.path.join("/content/resized", example["image"])}
        return example

    ds = Dataset.from_pandas(df)
    return ds.map(load_image)

train_ds = df_to_dataset(train_df)
val_ds   = df_to_dataset(val_df)

# 3) Processor ve model
model_id  = "google/paligemma-3b-pt-224"
processor = PaliGemmaProcessor.from_pretrained(model_id)

def collate_fn(examples):
    # 1) Görselleri PIL→RGB
    images  = [ to_pil(ex["image"]) for ex in examples ]
    # 2) Prompt’u hazırla
    prompts = ["<image> <bos> describe this image."] * len(examples)
    # 3) Hedef caption’ları al
    captions = [ ex["caption_1"] for ex in examples ]

    # 4) Tek çağrıda hem input_ids hem suffix (yani labels) oluştur
    batch = processor(
        text=prompts,
        images=images,
        suffix=captions,            # burası hedef label’ı veriyor
        return_tensors="pt",
        padding="longest",
        truncation=True
    )

    # 5) Pad token’ları -100 ile maskele (Trainer kayıpta bunları görmezden gelir)
    labels = batch["labels"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    batch["labels"] = labels

    return batch

    # Trainer’ın internal loss hesabı için labels ekle
    batch["labels"] = batch["input_ids"].clone()
    return batch

# 4) Model & LoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)
base_model = PaliGemmaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
lora_config = LoraConfig(
    r=8,
    target_modules=["q_proj","k_proj","v_proj","o_proj","up_proj","down_proj","gate_proj"],
    task_type="CAUSAL_LM"
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# 5) Eğitim argümanları
args = TrainingArguments(
    output_dir="finetuned_paligemma_riscm_yourdata",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    num_train_epochs=1,
    dataloader_pin_memory=False,
    bf16=True,
    logging_steps=10,
    max_steps=500,
    save_strategy="epoch",
    report_to="wandb",
    remove_unused_columns=False,
    dataloader_num_workers=8
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate_fn
)

# 6) Eğitim
trainer.train()

from time import time
from tqdm.auto import tqdm
import math
import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32   = True

# 1) Prepare model for fast generation
model.eval()
model.gradient_checkpointing_disable()   # re-enable use_cache

# 2) Inference in mini-batches with ETA
clean_preds, references = [], []
batch_size = 32
total_batches = math.ceil(len(val_ds) / batch_size)

pbar = tqdm(total=total_batches, desc="Generating", unit="batch")

for batch_idx in range(0, len(val_ds), batch_size):
    start_time = time()

    # select a true Dataset slice
    batch = val_ds.select(range(batch_idx, min(batch_idx+batch_size, len(val_ds))))
    images = [to_pil(ex["image"]) for ex in batch]
    refs   = [[ex[f"caption_{j}"] for j in range(1,6)] for ex in batch]
    prompts = ["<image> <bos> describe this image."] * len(images)

    inputs = processor(
        text=prompts,
        images=images,
        return_tensors="pt",
        padding="longest",
        truncation=True
    ).to("cuda")

    gen_ids = model.generate(**inputs, max_new_tokens=64)
    texts   = processor.tokenizer.batch_decode(gen_ids, skip_special_tokens=True)

    # record
    clean_preds.extend([t.strip() for t in texts])
    references.extend(refs)

    # update progress bar
    batch_time = time() - start_time
    pbar.update(1)
    pbar.set_postfix({"last_batch_s": f"{batch_time:.1f}"})

pbar.close()

# 3) Compute metrics (same as before)
refs_dict  = {i: references[i]    for i in range(len(references))}
preds_dict = {i: [clean_preds[i]] for i in range(len(clean_preds))}
cider_scorer = Cider()
cider_score, _ = cider_scorer.compute_score(refs_dict, preds_dict)

bleu   = evaluate.load("bleu"  ).compute(predictions=clean_preds, references=references)["bleu"]
meteor = evaluate.load("meteor").compute(predictions=clean_preds, references=references)["meteor"]
rouge  = evaluate.load("rouge" ).compute(predictions=clean_preds, references=references)["rougeL"]

metrics = {
    "eval/bleu":   bleu,
    "eval/meteor": meteor,
    "eval/rougeL": rouge,
    "eval/cider":  cider_score,
}
print(metrics)
wandb.log(metrics)

# 4) Save to CSV (same as before)
rows = []
for i, ex in enumerate(val_df.itertuples(index=False)):
    row = {"image": ex.image, "pred": clean_preds[i]}
    for j in range(1,6):
        row[f"ref_{j}"] = getattr(ex, f"caption_{j}")
    rows.append(row)
pd.DataFrame(rows).to_csv("inference_results.csv", index=False)


Map:   0%|          | 0/35614 [00:00<?, ? examples/s]

Map:   0%|          | 0/4453 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


trainable params: 11,298,816 || all params: 2,934,765,296 || trainable%: 0.3850


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to tru

NameError: Caught NameError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/_utils/worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/_utils/fetch.py", line 55, in fetch
    return self.collate_fn(data)
           ^^^^^^^^^^^^^^^^^^^^^
  File "<ipython-input-45-2895344672>", line 65, in collate_fn
    batch = processor(
            ^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/transformers/models/paligemma/processing_paligemma.py", line 313, in __call__
    labels = np.array(inputs["input_ids"])
             ^^
NameError: name 'np' is not defined
